# Imports

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# Sample Model Foraward Pass

In [2]:

def model(x: torch.Tensor, vocab_size:int, generator:torch.Generator=None) -> torch.Tensor:
    # x.shape = (batch_size, seq_len)

    B, T = x.shape
    logits = torch.randn(B, T, vocab_size, generator=generator, dtype=torch.float32)
    return logits


B, T = 1, 4
g = torch.Generator().manual_seed(42)
x = torch.randint(0, 26, (B, T), generator=g)
logits = model(x, vocab_size=27, generator=g)
print(x.shape, logits.shape)
print('-'*50)
print(x)
print('-'*50)
print(logits)
print('-'*50)

torch.Size([1, 4]) torch.Size([1, 4, 27])
--------------------------------------------------
tensor([[18, 21,  6,  4]])
--------------------------------------------------
tensor([[[ 0.6784, -1.2345, -0.0431, -1.6047,  1.7878, -0.4780, -0.2429,
          -0.9342, -0.7279, -0.5594, -0.7688,  0.7624, -1.5673, -0.2394,
           2.3228, -0.9634, -0.7581,  1.0783,  0.8008,  1.6806,  0.3559,
          -0.6866, -0.4934,  0.2415, -0.2316,  0.0418, -0.2516],
         [ 0.8599, -0.3097, -0.3957,  0.8034, -0.6216,  0.3189, -0.4245,
           0.3057, -0.7746,  0.0349,  0.3211,  1.5736, -0.8455, -1.2742,
           2.1228, -1.2347, -0.4879, -1.4181,  0.8963,  0.0499,  2.2667,
          -0.4880,  1.1914, -0.8140, -0.7360, -0.8371, -0.9224],
         [ 1.8113,  0.1606, -0.0978,  1.8446, -1.1845,  1.3835, -1.2024,
           0.7078, -1.0759,  0.5357,  0.3466, -0.1973, -1.0546,  1.2780,
           0.1453,  0.2311,  0.0087, -0.1423,  0.5750, -0.6417, -2.2064,
          -0.7508,  2.8140,  0.3598, -0.08

# Get Topk Probs - Rough

In [3]:
probs_test = torch.tensor([[[3, 9, 7, 1]], [[7, 2, 7, 9]]], dtype=torch.float32)
print('probs_test.shape:', probs_test.shape)
print('probs_test:\n', probs_test)
print('-'*50)

print('At inference, we only deal with the last token in the sequence, so we can drop the dim=1 from the above tensor')
probs_test = probs_test.squeeze(1)
print('probs_test.shape:', probs_test.shape)
print('probs_test:\n', probs_test)
print('-'*50)

k = 2
probs_test_args = torch.argsort(probs_test, dim=-1, descending=True)
print('probs_test_args.shape:', probs_test_args.shape)
print('probs_test_args:\n', probs_test_args)
print('-'*50)

probs_test_args_topk = probs_test_args[:, :k]
print('probs_test_args_topk:\n', probs_test_args_topk)

probs_test.shape: torch.Size([2, 1, 4])
probs_test:
 tensor([[[3., 9., 7., 1.]],

        [[7., 2., 7., 9.]]])
--------------------------------------------------
At inference, we only deal with the last token in the sequence, so we can drop the dim=1 from the above tensor
probs_test.shape: torch.Size([2, 4])
probs_test:
 tensor([[3., 9., 7., 1.],
        [7., 2., 7., 9.]])
--------------------------------------------------
probs_test_args.shape: torch.Size([2, 4])
probs_test_args:
 tensor([[1, 2, 0, 3],
        [3, 0, 2, 1]])
--------------------------------------------------
probs_test_args_topk:
 tensor([[1, 2],
        [3, 0]])


In [4]:
mask = torch.zeros_like(probs_test_args, dtype=torch.float32)
print(mask.shape)
print('-'*50)

for b in range(len(mask)):
    mask[b][probs_test_args_topk[b]] = 1
print(mask==0)
print('-'*50)

probs_test_topk_masked = probs_test.masked_fill(mask==0, value=-torch.inf)
print(probs_test_topk_masked)
print('-'*50)

torch.Size([2, 4])
--------------------------------------------------
tensor([[ True, False, False,  True],
        [False,  True,  True, False]])
--------------------------------------------------
tensor([[-inf, 9., 7., -inf],
        [7., -inf, -inf, 9.]])
--------------------------------------------------


In [5]:
torch.multinomial(
    F.softmax(probs_test_topk_masked, dim=-1), 
    num_samples=1, 
    replacement=True, 
    generator=g
)

tensor([[1],
        [3]])

# Cleaned Version:

- Given a single prompt of T tokens, generate x additional tokens in AR format & generate p different sample from the same token
    - Input Shape : (1,T)
    - Output Shape: (p,T+x)

In [6]:
# Dummy tokenizer
s2i = dict(zip([chr(i) for i in range(97, 123)], list(range(27))))
s2i[' '] = 26

i2s = dict(zip(list(range(27)), [chr(i) for i in range(97, 123)]))
i2s[26] = ' '

for i in range(27):
    assert s2i[i2s[i]] == i

In [7]:
# Dummy input prompt
prompt = 'emma'
x_tokens = torch.tensor([s2i[ch] for ch in list(prompt)]).unsqueeze(dim=0)
print(list(prompt))
print(x_tokens)
print(x_tokens.shape)


['e', 'm', 'm', 'a']
tensor([[ 4, 12, 12,  0]])
torch.Size([1, 4])


In [8]:
# Dummy Forward Pass
vocab_size = len(s2i)
logits = model(x_tokens, vocab_size, generator=g)
print(logits.shape)
print(logits)

torch.Size([1, 4, 27])
tensor([[[-0.9727,  0.9585,  1.6192,  1.4506,  0.2695, -0.2104, -0.7328,
           0.1043,  0.3488,  0.9676, -0.4657,  1.6048, -2.4801, -0.4175,
          -1.1955,  0.8123, -1.9006,  0.2286,  0.0249, -0.3460,  0.2868,
          -0.7308,  0.1748, -1.0939, -1.6022,  1.3529,  1.2888],
         [ 0.0523, -1.5469,  0.7567,  0.7755,  2.0265,  0.0358,  0.1206,
          -0.8057, -0.2076, -0.9319, -1.5910, -1.1360, -0.5226, -0.5188,
          -1.5013, -1.9267,  0.1279,  1.0229, -0.5558,  0.7043,  0.7099,
           1.7744, -0.9216,  0.9624, -0.3370, -1.1753,  0.3581],
         [ 0.4788,  1.3537,  0.5261,  2.1120, -0.5208, -0.9320,  0.1852,
           1.0687,  1.3065,  0.4598, -0.8146, -1.0212, -0.4949, -0.5923,
           0.1543,  0.4408, -0.1483, -2.3184, -0.3980,  1.0805, -1.7809,
           1.5080,  0.3094, -0.5003,  1.0350,  1.6896, -0.0045],
         [ 1.6668,  0.1539, -1.0603, -0.5727,  0.0836,  0.3999,  1.9892,
          -0.0720, -0.9061, -2.0487, -1.0811,  0.642

In [9]:
# Expand x_tokens for p samples
p = 5
x_tokens = x_tokens.expand(p, -1)
print(x_tokens.shape)
print(x_tokens)

torch.Size([5, 4])
tensor([[ 4, 12, 12,  0],
        [ 4, 12, 12,  0],
        [ 4, 12, 12,  0],
        [ 4, 12, 12,  0],
        [ 4, 12, 12,  0]])


In [10]:
# Run model forward pass
g = torch.Generator().manual_seed(42)
logits = model(x_tokens, vocab_size=vocab_size, generator=g)
print("logits.shape:", logits.shape)
print("logits:", logits)

logits.shape: torch.Size([5, 4, 27])
logits: tensor([[[ 1.9269e+00,  1.4873e+00,  9.0072e-01, -2.1055e+00,  6.7842e-01,
          -1.2345e+00, -4.3067e-02, -1.6047e+00, -7.5214e-01,  1.6487e+00,
          -3.9248e-01, -1.4036e+00, -7.2788e-01, -5.5943e-01, -7.6884e-01,
           7.6245e-01,  1.6423e+00, -1.5960e-01, -4.9740e-01,  4.3959e-01,
          -7.5813e-01,  1.0783e+00,  8.0080e-01,  1.6806e+00,  1.2791e+00,
           1.2964e+00,  6.1047e-01],
         [ 1.3347e+00, -2.3162e-01,  4.1759e-02, -2.5158e-01,  8.5986e-01,
          -1.3847e+00, -8.7124e-01, -2.2337e-01,  1.7174e+00,  3.1888e-01,
          -4.2452e-01,  3.0572e-01, -7.7459e-01, -1.5576e+00,  9.9564e-01,
          -8.7979e-01, -6.0114e-01, -1.2742e+00,  2.1228e+00, -1.2347e+00,
          -4.8791e-01, -9.1382e-01, -6.5814e-01,  7.8024e-02,  5.2581e-01,
          -4.8799e-01,  1.1914e+00],
         [-8.1401e-01, -7.3599e-01, -1.4032e+00,  3.6004e-02, -6.3477e-02,
           6.7561e-01, -9.7807e-02,  1.8446e+00, -1.1845

In [11]:
# Get the highest k logits per sample
k = 4

# Step1: Only pluck out the last token per sample - step1
logits_last = logits[:, -1, :]
print('logits_last.shape:', logits_last.shape)
print('logits_last:\n', logits_last)
print('-'*50)

# Step2: Argsort
logits_last_argsort = torch.argsort(logits_last, dim=-1, descending=True)
print('logits_last_argsort.shape:', logits_last_argsort.shape)
print('logits_last_argsort:\n', logits_last_argsort)
print('-'*50)

# Step3: Get topk idxs
logits_last_argsort_topk = logits_last_argsort[:, :k]
print('logits_last_argsort_topk:\n', logits_last_argsort_topk)
print('-'*50)


logits_last.shape: torch.Size([5, 27])
logits_last:
 tensor([[-0.3387, -1.3407, -0.5854,  0.5362,  0.5246,  1.1412,  0.0516,  0.7440,
         -0.4816, -1.0495,  0.6039, -1.7223, -0.8278,  1.3347,  0.4835, -2.5095,
          0.4880,  0.7846,  0.0286,  0.6408,  0.5832,  1.0669, -0.4502, -0.1853,
          0.7528,  0.4048,  0.1785],
        [-0.5558,  0.7043,  0.7099,  1.7744, -0.9216,  0.9624, -0.3370, -1.1753,
          0.3581,  0.4788,  1.3537,  0.5261,  2.1120, -0.5208, -0.9320,  0.1852,
          1.0687,  1.3065,  0.4598, -0.8146, -1.0212, -0.4949, -0.5923,  0.1543,
          0.4408, -0.1483, -2.3184],
        [ 0.2539,  0.9364,  0.7122, -0.0318,  0.1016,  1.3433,  0.7133,  0.4038,
         -0.7140,  0.8337, -0.9585,  0.4536,  1.2461, -2.3065, -1.2869,  0.1799,
         -2.1268, -0.1341, -1.0408, -0.7647, -0.0553,  1.2049, -0.9825,  0.4334,
         -0.7172,  1.0554, -1.4534],
        [-0.1426,  0.1527, -0.0388,  0.9446, -1.5824,  0.9871,  1.1457, -0.1418,
         -0.2763, -0.1932,

In [12]:
# Step4.1: Sprinkle -inf to the other idxs other than the topk
mask = torch.zeros_like(logits_last)
for i in range(len(logits_last_argsort_topk)):
    mask[i][logits_last_argsort_topk[i]] = 1
print('mask.shape:', mask.shape)
print('mask:\n', mask)

# Step4.2: Sprinkle -inf to the other idxs other than the topk
logits_last_topk_masked = logits_last.masked_fill(mask==0, value=-torch.inf)
print('logits_last_topk_masked.shape:', logits_last_topk_masked.shape)
print('logits_last_topk_masked:\n', logits_last_topk_masked)


mask.shape: torch.Size([5, 27])
mask:
 tensor([[0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 1.,
         0., 0., 0., 1., 0., 0., 0., 0., 0.],
        [0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 1., 0., 1., 0., 0., 0., 0., 1.,
         0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0.,
         0., 0., 0., 1., 0., 0., 0., 1., 0.],
        [0., 0., 0., 1., 0., 1., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 1., 0., 0., 0.],
        [0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 1., 0., 1., 1.]])
logits_last_topk_masked.shape: torch.Size([5, 27])
logits_last_topk_masked:
 tensor([[  -inf,   -inf,   -inf,   -inf,   -inf, 1.1412,   -inf,   -inf,   -inf,
           -inf,   -inf,   -inf,   -inf, 1.3347,   -inf,   -inf,   -inf, 0.7846,
           -inf,   -inf,   -inf, 1.0669,   -inf,   -inf,   -inf,   -inf,   -inf],
    

In [13]:
# Get Softmax
probs_last = F.softmax(logits_last_topk_masked, dim=-1)
print('probs_last.shape:', probs_last.shape)
print('probs_last:\n', probs_last)
print('-'*50)

# Sample
new_tokens = torch.multinomial(probs_last, num_samples=1)
print('new_tokens.shape:', new_tokens.shape)
print('new_tokens:\n', new_tokens)
print('-'*50)

probs_last.shape: torch.Size([5, 27])
probs_last:
 tensor([[0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.2603, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.3159, 0.0000, 0.0000, 0.0000, 0.1822,
         0.0000, 0.0000, 0.0000, 0.2417, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.2714, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.1782, 0.0000, 0.3804, 0.0000, 0.0000, 0.0000, 0.0000, 0.1700,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.2835, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.2572, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.2468, 0.0000, 0.0000, 0.0000, 0.2125, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.2293, 0.0000, 0.2393, 0.2804, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.2510

In [14]:
# Concat
torch.cat([x_tokens, new_tokens], dim=-1)

tensor([[ 4, 12, 12,  0, 13],
        [ 4, 12, 12,  0,  3],
        [ 4, 12, 12,  0, 25],
        [ 4, 12, 12,  0,  6],
        [ 4, 12, 12,  0,  3]])

# Clean Code: Refactored

In [15]:
def model(x: torch.Tensor, vocab_size:int, generator:torch.Generator=None) -> torch.Tensor:
    # x.shape = (batch_size, seq_len)

    B, T = x.shape
    logits = torch.randn(B, T, vocab_size, generator=generator, dtype=torch.float32)
    return logits


def generate_samples(x_tokens:torch.Tensor, num_samples:int, num_tokens:int, topk_logits:int):
    # x.shape: B, T

    # Expand x_tokens to (B * num_samples, T)
    x_tokens = x_tokens.expand(num_samples, -1)

    g = torch.Generator().manual_seed(42)
    for i in range(num_tokens):

        # Model Forward Pass
        logits = model(x_tokens, vocab_size=27, generator=g)

        logits_last = logits[:, -1, :]
        logits_last_argsort = torch.argsort(logits_last, dim=-1, descending=True)
        logits_last_argsort_topk = logits_last_argsort[:, :topk_logits]

        mask = torch.zeros_like(logits_last_argsort)
        for j in range(len(logits_last_argsort_topk)):
            mask[j][logits_last_argsort_topk[j]] = 1
        
        logits_last_topk_masked = logits_last.masked_fill(mask==0, value=-torch.inf)
        
        # Sample from the top-k logits
        probs_last = F.softmax(logits_last_topk_masked, dim=-1)
        next_token = torch.multinomial(probs_last, num_samples=1)
        
        # Append the next token to the sequence
        x_tokens = torch.cat([x_tokens, next_token], dim=-1)

    return x_tokens

        

# Dummy input prompt
prompt = 'emma'
x_tokens = torch.tensor([s2i[ch] for ch in list(prompt)]).unsqueeze(dim=0)
print(list(prompt))
print(x_tokens)
print(x_tokens.shape)
print('-'*50)

# Generate samples
num_samples = 5
num_tokens = 10
topk_logits = 6
samples = generate_samples(x_tokens, num_samples, num_tokens, topk_logits)
print(samples)
print('-'*50)

# Decode samples
# [i2s[ch.item()] for sample in samples for ch in sample]
sample_char = [[i2s[ch.item()] for ch in sample] for sample in samples]
print('sample_char:\n', sample_char)
print('-'*50)

# Final samples
ans = [('').join(sample) for sample in sample_char]
print('ans:\n', ans)


['e', 'm', 'm', 'a']
tensor([[ 4, 12, 12,  0]])
torch.Size([1, 4])
--------------------------------------------------
tensor([[ 4, 12, 12,  0, 24,  8, 18, 25, 16,  4,  0, 10,  7,  6],
        [ 4, 12, 12,  0, 17, 24,  6,  6, 18,  9,  0,  1, 20,  5],
        [ 4, 12, 12,  0, 12, 23, 18, 13, 12,  8, 19, 17,  5,  8],
        [ 4, 12, 12,  0, 10,  1, 22, 14, 20, 25,  8,  2,  9,  6],
        [ 4, 12, 12,  0, 23,  8,  3, 10,  4, 23, 19,  1, 20,  6]])
--------------------------------------------------
sample_char:
 [['e', 'm', 'm', 'a', 'y', 'i', 's', 'z', 'q', 'e', 'a', 'k', 'h', 'g'], ['e', 'm', 'm', 'a', 'r', 'y', 'g', 'g', 's', 'j', 'a', 'b', 'u', 'f'], ['e', 'm', 'm', 'a', 'm', 'x', 's', 'n', 'm', 'i', 't', 'r', 'f', 'i'], ['e', 'm', 'm', 'a', 'k', 'b', 'w', 'o', 'u', 'z', 'i', 'c', 'j', 'g'], ['e', 'm', 'm', 'a', 'x', 'i', 'd', 'k', 'e', 'x', 't', 'b', 'u', 'g']]
--------------------------------------------------
ans:
 ['emmayiszqeakhg', 'emmaryggsjabuf', 'emmamxsnmitrfi', 'emmakbwouzic

# Efficient Top-k Sampling in PyTorch

When implementing LLM inference, the model forward pass is usually the most expensive operation. However, there are also a few PyTorch operations that can unnecessarily slow down sampling if implemented naively.

---

# 1. Avoid `torch.argsort()` → Use `torch.topk()`

## ❌ Inefficient

```python
logits = torch.tensor([[2.1, 4.3, 1.2, 5.8, 3.6]])

idx = torch.argsort(logits, dim=-1, descending=True)
top3 = idx[:, :3]
```

Output:

```
argsort:
[[3, 1, 4, 0, 2]]

top3:
[[3, 1, 4]]
```

### What's happening?

`argsort()` sorts **every element**.

For a vocabulary of

```
V = 50,000
```

and

```
top_k = 40
```

it still sorts all **50,000 logits**, even though we only keep 40.

Complexity is roughly

```
O(V log V)
```

---

## ✅ Efficient

```python
values, indices = torch.topk(logits, k=3, dim=-1)
```

Output:

```
values:
[[5.8, 4.3, 3.6]]

indices:
[[3, 1, 4]]
```

Only the largest `k` elements are found.

Complexity is approximately

```
O(V)
```

This is what essentially all production LLM inference libraries use.

---

# 2. Avoid Python loops for masking

Suppose we have

```python
logits = torch.tensor([
    [2.1, 4.3, 1.2, 5.8, 3.6],
    [0.5, 6.0, 3.2, 2.1, 8.4]
])
```

Top-2 indices

```
[[3,1],
 [4,1]]
```

---

## ❌ Inefficient

```python
mask = torch.zeros_like(logits)

for i in range(len(topk_idx)):
    mask[i][topk_idx[i]] = 1
```

Result

```
mask

0 1 0 1 0
0 1 0 0 1
```

Problems

- Python loop
- Runs once per batch element
- Doesn't leverage PyTorch's optimized tensor operations

---

## ✅ Efficient

```python
mask = torch.zeros_like(logits, dtype=torch.bool)

mask.scatter_(
    dim=1,
    index=topk_idx,
    value=True
)
```

Result

```
mask

False True  False True  False
False True  False False True
```

Everything is done inside optimized PyTorch kernels.

---

# 3. `scatter_()` intuition

Think of `scatter_()` as

> "Place these values at these indices."

Suppose

```python
values = torch.tensor([
    [10,20],
    [30,40]
])

indices = torch.tensor([
    [1,3],
    [0,2]
])
```

Start with

```python
output = torch.zeros(2,5)
```

```
0 0 0 0 0
0 0 0 0 0
```

Now

```python
output.scatter_(1, indices, values)
```

Result

```
0 10 0 20 0
30 0 40 0 0
```

The values are "scattered" into the output tensor at the specified indices.

---

# 4. Top-k masking using `scatter_()`

Suppose

```
logits

2.1 4.3 1.2 5.8 3.6
```

Top-2

```
values

5.8 4.3

indices

3 1
```

Create

```python
filtered = torch.full_like(logits, -torch.inf)
```

```
-inf -inf -inf -inf -inf
```

Scatter

```python
filtered.scatter_(1, indices, values)
```

Result

```
-inf 4.3 -inf 5.8 -inf
```

Now

```python
probs = F.softmax(filtered, dim=-1)
```

Only the top-k logits receive non-zero probability.

No explicit mask is needed.

---

# Summary

| Don't | Do Instead | Why |
|--------|------------|-----|
| `torch.argsort()` | `torch.topk()` | Finds only the top-k elements without sorting the full vocabulary |
| Python loop to build masks | `scatter_()` | Fully vectorized, faster, cleaner |
| Manual indexing | `scatter_()` | Leverages optimized PyTorch kernels |
| Sort entire vocabulary | Select only top-k | Reduces unnecessary computation |

---

# Rule of Thumb

Whenever you find yourself writing

```python
for i in range(batch_size):
    ...
```

ask yourself:

> **"Can this be expressed as a PyTorch tensor operation instead?"**

In many cases, operations like `topk()`, `scatter_()`, `gather()`, and advanced indexing eliminate Python loops while making the code both cleaner and significantly faster.

# PyTorch `scatter_()` — A Practical Guide - V1

`scatter_()` is a tensor operation used to **write values into a tensor at specified indices**.

It is especially useful in Deep Learning for:

- One-hot encoding
- Top-k filtering
- Classification targets
- Token selection
- Routing / Mixture-of-Experts
- Replacing Python loops with vectorized tensor operations

The basic form is:

```python
self.scatter_(dim, index, src)
```

Think:

```text
self  ──► the tensor we are WRITING INTO
index ──► tells us WHERE to write
src   ──► tells us WHAT to write
dim   ──► tells us WHICH DIMENSION of `self` the indices refer to
```

> **Important:** `dim` refers to a dimension of **`self`**, not `src` or `index`.

---

# 1. Understanding the Shapes

The most important thing to remember is:

> `self`, `index`, and `src` do **not** necessarily have the same shape, but they must satisfy specific dimensionality/size constraints.

For:

```python
self.scatter_(dim, index, src)
```

### Rule 1 — `index` must have the same number of dimensions as `self`

For example:

```python
self.shape  = (2, 5)
index.shape = (2, 2)
```

is valid because both are 2D.

But:

```python
self.shape  = (2, 5)
index.shape = (2,)
```

is **not valid**.

You may encounter:

```text
Index tensor must have the same number of dimensions as self tensor
```

This is a common error.

If you have:

```python
labels.shape == (B,)
```

but need to scatter into:

```python
one_hot.shape == (B, V)
```

you therefore do:

```python
labels = labels.unsqueeze(1)
```

giving:

```text
labels:          (B,)
                   ↓ unsqueeze
labels:          (B, 1)
```

Now both `self` and `index` are 2D.

---

### Rule 2 — `index` specifies the size of the output region being written

For example:

```python
self.shape  = (2, 5)
index.shape = (2, 2)
src.shape   = (2, 2)
```

is valid:

```text
self:

0  0  0  0  0
0  0  0  0  0

index:

1  3
0  2

src:

10  20
30  40
```

With:

```python
self.scatter_(1, index, src)
```

we write:

```text
0  10  0  20  0
30  0  40  0  0
```

Notice:

```text
self  = (2, 5)
index = (2, 2)
src   = (2, 2)
```

`self` does **not** need to have the same shape as `index` or `src`.

---

### Rule 3 — `index` and `src` must be compatible

The simplest and most common pattern is:

```text
index.shape == src.shape
```

For example:

```python
index.shape = (B, K)
src.shape   = (B, K)
```

This is exactly what we have in the LLM top-k example:

```python
topk_values.shape = (B, K)
topk_indices.shape = (B, K)
```

and:

```python
filtered.shape = (B, V)
```

Then:

```python
filtered.scatter_(
    dim=1,
    index=topk_indices,
    src=topk_values
)
```

---

### Rule 4 — `dim` refers to `self`

Suppose:

```python
self.shape = (B, V)
```

Then:

```text
dim=0 → batch dimension
dim=1 → vocabulary dimension
```

So:

```python
self.scatter_(dim=1, ...)
```

means:

> "Use the indices to choose **columns of `self`**."

For:

```python
self.shape = (B, T, V)
```

we have:

```text
dim=0 → batch
dim=1 → sequence
dim=2 → vocabulary
```

Therefore:

```python
self.scatter_(dim=2, ...)
```

means:

> "Use the indices to choose positions along the vocabulary dimension of `self`."

---

### A useful mental model

Always start with **`self`**:

```text
self.shape = (B, T, V)

               ↓
       Which dimension?
               ↓
dim=0      dim=1      dim=2
 batch    sequence   vocabulary
```

Then ask:

> **What do my `index` values refer to along that dimension?**

This makes choosing `dim` much easier.

---

# PyTorch `scatter_()` — A Practical Guide - V2

`scatter_()` is a tensor operation used to **write values into a tensor at specified indices**.

It is especially useful in Deep Learning for:

- One-hot encoding
- Top-k filtering
- Classification targets
- Token selection
- Routing / Mixture-of-Experts
- Replacing Python loops with vectorized tensor operations

---

# 1. The basic idea

The simplest form is:

```python
output.scatter_(dim, index, src)
```

Think:

```text
output
   ↑
   │
"put these values"
   │
   │
index ──► tells us WHERE
src   ──► tells us WHAT
```

The most important argument is:

```python
dim
```

It tells PyTorch:

> **Along which dimension should I interpret the indices?**

---

# 2. 1D example

Start with:

```python
x = torch.zeros(5)

index = torch.tensor([1, 3])
src = torch.tensor([10., 20.])

x.scatter_(0, index, src)
```

Before:

```text
x

0  0  0  0  0
```

`index` says:

```text
put 10 at position 1
put 20 at position 3
```

After:

```text
0  10  0  20  0
```

Here:

```python
dim=0
```

because a 1D tensor has only one dimension.

---

# 3. Why does `dim` matter?

Consider a 2D tensor:

```python
x = torch.zeros(2, 4)
```

Visualize it as:

```text
        dim=1 →
      0  0  0  0
dim=0
  ↓   0  0  0  0
```

There are two dimensions:

```text
dim=0 → rows
dim=1 → columns
```

Therefore:

```python
scatter_(0, ...)
```

means:

> Scatter along the **rows** dimension.

while:

```python
scatter_(1, ...)
```

means:

> Scatter along the **columns** dimension.

This distinction is the most important thing to understand.

---

# 4. `dim=1`: scatter into columns

Consider:

```python
x = torch.zeros(2, 4)

index = torch.tensor([
    [1, 3],
    [0, 2]
])

src = torch.tensor([
    [10., 20.],
    [30., 40.]
])

x.scatter_(1, index, src)
```

We have:

```text
x

0  0  0  0
0  0  0  0
```

For row 0:

```text
index = [1, 3]
src   = [10, 20]
```

So:

```text
row 0:

put 10 at column 1
put 20 at column 3
```

Result:

```text
0  10  0  20
```

For row 1:

```text
index = [0, 2]
src   = [30, 40]
```

Result:

```text
30  0  40  0
```

Final:

```text
0  10   0  20
30  0  40   0
```

### Mental model

For:

```python
scatter_(dim=1)
```

each **row gets its own indices**.

This is extremely common in Deep Learning.

---

# 5. `dim=0`: scatter into rows

Now let's use the same tensor but:

```python
x = torch.zeros(2, 4)

index = torch.tensor([
    [1, 1],
    [0, 0]
])

src = torch.tensor([
    [10., 20.],
    [30., 40.]
])

x.scatter_(0, index, src)
```

Here:

```python
dim=0
```

means the indices refer to the **row dimension**.

Visualize:

```text
       columns
       ↓   ↓
row 0   0   0   0   0
row 1   0   0   0   0
  ↑
 dim=0
```

The indices tell us which **row** to write into.

For column 0:

```text
index = [1, 0]
src   = [10, 30]
```

So:

```text
row 1, column 0 ← 10
row 0, column 0 ← 30
```

Similarly for column 1.

Result:

```text
30  40  0  0
10  20  0  0
```

---

# 6. The most important DL example: One-hot encoding

Suppose we have 4 classification targets:

```python
targets = torch.tensor([2, 0, 3, 1])
```

There are 5 classes.

We want:

```text
class 2 → [0 0 1 0 0]
class 0 → [1 0 0 0 0]
class 3 → [0 0 0 1 0]
class 1 → [0 1 0 0 0]
```

Create:

```python
one_hot = torch.zeros(4, 5)
```

Shape:

```text
(batch_size, num_classes)
        ↓           ↓
        4           5
```

We want to place `1` into each row at the target class.

```python
one_hot.scatter_(
    dim=1,
    index=targets.unsqueeze(1),
    value=1
)
```

Result:

```text
0 0 1 0 0
1 0 0 0 0
0 0 0 1 0
0 1 0 0 0
```

### Why `dim=1`?

Because:

```text
dim=0 → batch
dim=1 → classes
```

We want the target index to select a **class within each batch row**.

So:

```python
dim=1
```

---

# 7. Top-k masking — LLM inference

This is the example relevant to LLM generation.

Suppose:

```python
logits = torch.tensor([
    [2.1, 4.3, 1.2, 5.8, 3.6],
    [0.5, 6.0, 3.2, 2.1, 8.4]
])
```

Shape:

```text
(B, vocab_size)

(2, 5)
```

Find top-2:

```python
values, indices = torch.topk(logits, k=2, dim=1)
```

Result:

```text
values:

5.8  4.3
8.4  6.0
```

and:

```text
indices:

3  1
4  1
```

We want:

```text
row 0:
-inf  4.3  -inf  5.8  -inf

row 1:
-inf  6.0  -inf  -inf  8.4
```

Create:

```python
filtered = torch.full_like(logits, -torch.inf)
```

Then:

```python
filtered.scatter_(
    dim=1,
    index=indices,
    src=values
)
```

Result:

```text
-inf   4.3  -inf   5.8  -inf
-inf   6.0  -inf  -inf   8.4
```

Then:

```python
probs = F.softmax(filtered, dim=1)
```

Only the top-2 tokens have non-zero probability.

### Why `dim=1`?

Because the tensor is:

```text
(B, V)

dim=0 → batch
dim=1 → vocabulary
```

The top-k indices refer to **vocabulary positions within each batch element**.

Therefore:

```python
scatter_(dim=1, ...)
```

---

# 8. `scatter_()` vs `gather()`

These two are often confused.

### `scatter`

```text
values → positions
```

You tell PyTorch:

> Put these values HERE.

```python
output.scatter_(dim, index, values)
```

### `gather`

```text
positions → values
```

You tell PyTorch:

> Give me the values FROM HERE.

```python
output.gather(dim, index)
```

A useful mental pair:

```text
scatter = WRITE
gather  = READ
```

---

# 9. A simple scatter/gather pair

Start:

```python
x = torch.tensor([
    [10, 20, 30, 40],
    [50, 60, 70, 80]
])
```

Suppose:

```python
index = torch.tensor([
    [3, 1],
    [0, 2]
])
```

### Gather

```python
x.gather(1, index)
```

gives:

```text
40 20
50 70
```

We **read** values from positions.

---

### Scatter

```python
out = torch.zeros_like(x)

out.scatter_(1, index, torch.tensor([
    [400, 200],
    [500, 700]
]))
```

gives:

```text
0   200   0   400
500   0  700    0
```

We **write** values into positions.

---

# 10. A useful visualization for `dim`

For a matrix:

```text
          dim=1
       → → → → →

dim=0   [ 1  2  3  4 ]
  ↓     [ 5  6  7  8 ]
        [ 9 10 11 12 ]
```

Remember:

```text
dim=0 → move DOWN rows
dim=1 → move ACROSS columns
```

For DL tensors, extend this idea:

```text
(B, T, V)

dim=0 → batch
dim=1 → sequence
dim=2 → vocabulary
```

So if you have:

```python
logits.shape == (B, T, V)
```

and want to scatter into the vocabulary dimension:

```python
logits.scatter_(dim=2, ...)
```

---

# 11. Common DL patterns

## One-hot encoding

```python
one_hot.scatter_(
    1,
    labels.unsqueeze(1),
    1
)
```

```text
(B, C)
 ↑   ↑
batch classes
```

---

## LLM top-k filtering

```python
filtered.scatter_(
    1,
    topk_indices,
    topk_values
)
```

```text
(B, V)
 ↑   ↑
batch vocab
```

---

## Token selection

For a tensor:

```python
x.shape == (B, T, D)
```

if selecting/scattering across sequence positions:

```python
dim=1
```

because:

```text
B → dim 0
T → dim 1
D → dim 2
```

---

# 12. The key rule

When you see:

```python
x.scatter_(dim, index, src)
```

ask two questions:

### 1. What does `dim` represent?

For example:

```text
(B, V)

dim=0 → batch
dim=1 → vocabulary
```

### 2. What do the indices refer to?

If:

```python
index = [[3, 1],
         [4, 1]]
```

and:

```python
dim=1
```

then these are **column/vocabulary indices for each row**.

---

# Cheat Sheet

```text
scatter = WRITE values at indices
gather  = READ values at indices
```

For a 2D tensor:

```text
        dim=1 →
      0  0  0  0
dim=0
  ↓   0  0  0  0
```

Typical DL tensors:

```text
(B, C)
 ↑   ↑
 0   1

(B, T, D)
 ↑   ↑   ↑
 0   1   2

(B, T, V)
 ↑   ↑   ↑
 0   1   2
```

Common uses:

```python
# One-hot
one_hot.scatter_(1, labels.unsqueeze(1), 1)

# LLM top-k filtering
filtered.scatter_(1, topk_indices, topk_values)

# General rule
output.scatter_(dim, index, values)
```

The most useful mental model is:

> **`dim` tells you which axis the indices are selecting along.**
>
> **`index` tells you where to write.**
>
> **`src` tells you what to write.**

In [28]:
# torch.topk()
test_a = torch.randint(0, 10, (4, 6))
print(test_a)
print('-'*50)
test_a_topk_idxs = torch.topk(test_a, k=3, dim=-1).indices
print(test_a_topk_idxs)
print('-'*50)

tensor([[0, 2, 8, 6, 1, 4],
        [6, 2, 8, 0, 0, 9],
        [5, 8, 5, 7, 3, 1],
        [3, 1, 6, 1, 7, 9]])
--------------------------------------------------
tensor([[2, 3, 5],
        [5, 2, 0],
        [1, 3, 2],
        [5, 4, 2]])
--------------------------------------------------


In [44]:
# This is a case of wrong OHE
targets=torch.tensor([2, 3, 4, 1])
test_ohe = torch.zeros(5, 5)
print(test_ohe.scatter_(dim=0, index=targets.unsqueeze(dim=0), value=1))


try:
    # This is a case of wrong OHE - here at 4, we are saying we want to inject a 1 in the 3rd column and 5th row (idx=4) of self whihc does not exist
    targets=torch.tensor([2, 3, 4, 1])
    test_ohe = torch.zeros(4, 5)
    print(test_ohe.scatter_(dim=0, index=targets.unsqueeze(dim=0), value=1))
except Exception as e:
    print(e)

tensor([[0., 0., 0., 0., 0.],
        [0., 0., 0., 1., 0.],
        [1., 0., 0., 0., 0.],
        [0., 1., 0., 0., 0.],
        [0., 0., 1., 0., 0.]])
index 4 is out of bounds for dimension 0 with size 4


In [16]:
# # Get the top k logits and their indices
# top_k_logits, top_k_indices = torch.topk(logits_last, k, dim=-1)

# # Create a mask for the top k logits
# mask = torch.zeros_like(logits_last)
# mask.scatter_(-1, top_k_indices, 1)

# # Apply the mask to the logits
# logits_last_masked = logits_last.masked_fill(mask == 0, float('-inf'))

# # Get the probabilities
# probs = F.softmax(logits_last_masked, dim=-1)

# # Sample from the probabilities
# next_token = torch.multinomial(probs, num_samples=1)

In [49]:
def model(x: torch.Tensor, vocab_size:int, generator:torch.Generator=None) -> torch.Tensor:
    # x.shape = (batch_size, seq_len)

    B, T = x.shape
    logits = torch.randn(B, T, vocab_size, generator=generator, dtype=torch.float32)
    return logits


def generate_samples_v2(x_tokens:torch.Tensor, num_samples:int, num_tokens:int, topk_logits:int):
    # x.shape: B, T

    # Expand x_tokens to (B * num_samples, T)
    x_tokens = x_tokens.expand(num_samples, -1)

    g = torch.Generator().manual_seed(42)
    for i in range(num_tokens):

        # Model Forward Pass
        logits = model(x_tokens, vocab_size=27, generator=g)

        logits_last = logits[:, -1, :]

        # Get highest k logits per batch
        topk_vals, topk_idxs = torch.topk(logits_last, k=topk_logits, dim=-1)

        # Mask out all but topk logits per batch
        filtered_logits = torch.full_like(logits_last, fill_value=-torch.inf)
        filtered_logits.scatter_(dim=1, index=topk_idxs, src=topk_vals) # In-place update

        # Sample from the top-k logits
        probs_last = F.softmax(mask, dim=-1)
        next_token = torch.multinomial(probs_last, num_samples=1)
        
        # Append the next token to the sequence
        x_tokens = torch.cat([x_tokens, next_token], dim=-1)

    return x_tokens

        

# Dummy input prompt
prompt = 'emma'
x_tokens = torch.tensor([s2i[ch] for ch in list(prompt)]).unsqueeze(dim=0)
print(list(prompt))
print(x_tokens)
print(x_tokens.shape)
print('-'*50)

# Generate samples
num_samples = 5
num_tokens = 10
topk_logits = 6
samples = generate_samples_v2(x_tokens, num_samples, num_tokens, topk_logits)
print(samples)
print('-'*50)

# # Decode samples
# # [i2s[ch.item()] for sample in samples for ch in sample]
# sample_char = [[i2s[ch.item()] for ch in sample] for sample in samples]
# print('sample_char:\n', sample_char)
# print('-'*50)

# # Final samples
# ans = [('').join(sample) for sample in sample_char]
# print('ans:\n', ans)


['e', 'm', 'm', 'a']
tensor([[ 4, 12, 12,  0]])
torch.Size([1, 4])
--------------------------------------------------
tensor([[ 4, 12, 12,  0, 21, 21, 20, 22,  2, 19, 13,  9,  7,  0],
        [ 4, 12, 12,  0,  3, 14, 15,  6, 18,  3,  2,  3, 20,  6],
        [ 4, 12, 12,  0,  5,  5, 25, 13, 23,  2,  2, 11,  3,  8],
        [ 4, 12, 12,  0, 23, 17, 13, 16, 24, 10, 19,  6, 14,  1],
        [ 4, 12, 12,  0, 23, 11,  1, 26, 25, 12,  8, 17,  0, 25]])
--------------------------------------------------


# Temperature

In [59]:
test_logits1 = torch.tensor([8, 4, 2, -2]).float()
test_softmax1 = F.softmax(test_logits1, dim=0)
print(test_softmax1)

temp = 2
test_logits2 = torch.tensor([8, 4, 2, -2]).float()/temp
test_softmax2 = F.softmax(test_logits2, dim=0)
print(test_softmax2)


temp = 20
test_logits3 = torch.tensor([8, 4, 2, -2]).float()/temp
test_softmax3 = F.softmax(test_logits3, dim=0)
print(test_softmax3)

temp = 0.2
test_logits4 = torch.tensor([8, 4, 2, -2]).float()/temp
test_softmax4 = F.softmax(test_logits4, dim=0)
print(test_softmax4)


print('Default setup is temp=1. Higher temp (>1) drives the ssoftmax o/p towards uniform distribution - more random. Lower temp (<1) makes the model more deterministic')


tensor([9.7959e-01, 1.7942e-02, 2.4282e-03, 4.4473e-05])
tensor([0.8390, 0.1135, 0.0418, 0.0057])
tensor([0.3158, 0.2586, 0.2340, 0.1916])
tensor([1.0000e+00, 2.0612e-09, 9.3576e-14, 1.9287e-22])
Default setup is temp=1. Higher temp (>1) drives the ssoftmax o/p towards uniform distribution - more random. Lower temp (<1) makes the model more deterministic
